In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

### bar analysis_NMF

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# =========================
# PATHS
# =========================

DATA_DIR = Path("/home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/data")

OUT_BASE = Path("/home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/NMF_bar_figures")
OUT_BASE.mkdir(parents=True, exist_ok=True)

# =========================
# STORAGE
# =========================

results = defaultdict(lambda: defaultdict(dict))

# =========================
# UNSANITIZE CATEGORY
# =========================

def unsanitize_category(sanitized, known_categories=None):
    """Reverse filename sanitization back to actual category name.
    E.g. 'Travel___Navigation' -> 'Travel & Navigation'
    """

    if known_categories:

        for cat in known_categories:

            s = cat.replace(" ", "_")
            s = re.sub(r"[^0-9A-Za-z_]+", "_", s)

            if s == sanitized:
                return cat

    # Fallback
    s = re.sub(r'___', ' & ', sanitized)
    s = s.replace('_', ' ')

    return s

# =========================
# PARSE FILENAME
# =========================

def parse_filename(fname):

    if fname.startswith("ORIGINAL"):

        k = int(re.search(r'(\d+)', fname).group(1))

        return "ORIGINAL", "original", k

    match = re.match(
        r"f_(.*?)_(\d+u)_.*?_(\d+)recommendation\.csv",
        fname
    )

    if match:

        category_raw = match.group(1)
        n = match.group(2)
        k = int(match.group(3))

        return category_raw, n, k

    return None, None, None

# =========================
# METRIC
# =========================

def compute_avg(df, target_category):

    df['has_category'] = (
        df['genre_g1']
        .fillna('')
        .astype(str)
        .str.lower()
        == target_category.lower()
    )

    user_counts = df.groupby('user_id')['has_category'].sum()

    return user_counts.mean(), len(user_counts)

# =========================
# COLLECT KNOWN CATEGORIES FROM ORIGINAL
# =========================

known_categories = set()

for fname in os.listdir(DATA_DIR):

    if fname.startswith("ORIGINAL") and fname.endswith(".csv"):

        try:

            df_tmp = pd.read_csv(
                DATA_DIR / fname,
                on_bad_lines='skip'
            )

        except Exception as e:

            print(f"Failed reading {fname}: {e}")

            continue

        print(f"{fname} columns:", df_tmp.columns.tolist())

        if 'genre_g1' not in df_tmp.columns:

            print(f"Skipping {fname} -> no 'genre_g1' column found")

            continue

        for c in df_tmp['genre_g1'].dropna().unique():

            c = str(c).strip()

            if c and c != "Unknown":
                known_categories.add(c)

        break  # One file is enough

print(f"Known categories: {known_categories}")

# =========================
# READ NON-ORIGINAL
# =========================

all_n_values = set()

for fname in os.listdir(DATA_DIR):

    if not fname.endswith(".csv"):
        continue

    category_raw, n, k = parse_filename(fname)

    if category_raw is None or n == "original":
        continue

    category = unsanitize_category(
        category_raw,
        known_categories
    )

    try:

        df = pd.read_csv(
            DATA_DIR / fname,
            on_bad_lines='skip'
        )

    except Exception as e:

        print(f"Failed reading {fname}: {e}")

        continue

    if 'genre_g1' not in df.columns:

        print(f"Skipping {fname} -> no 'genre_g1' column found")

        continue

    avg_val, num_users = compute_avg(df, category)

    results[category][k][n] = (
        avg_val,
        num_users
    )

    all_n_values.add(n)

# =========================
# ORIGINAL FILES
# =========================

original_data = defaultdict(dict)

for fname in os.listdir(DATA_DIR):

    if not fname.startswith("ORIGINAL"):
        continue

    _, _, k = parse_filename(fname)

    try:

        df = pd.read_csv(
            DATA_DIR / fname,
            on_bad_lines='skip'
        )

    except Exception as e:

        print(f"Failed reading {fname}: {e}")

        continue

    if 'genre_g1' not in df.columns:

        print(f"Skipping {fname} -> no 'genre_g1' column found")

        continue

    # Get all categories except Unknown
    all_cats = set(
        df['genre_g1']
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    all_cats.discard("Unknown")

    for cat in all_cats:

        avg_val, num_users = compute_avg(
            df.copy(),
            cat
        )

        original_data[cat][k] = (
            avg_val,
            num_users
        )

# =========================
# MERGE ORIGINAL
# =========================

for category in results.keys():

    for k in results[category].keys():

        if (
            category in original_data
            and k in original_data[category]
        ):

            results[category][k]['original'] = (
                original_data[category][k]
            )

# =========================
# COLORS (FINAL STYLE PALETTE)
# =========================

def generate_colors(n_list):

    palette = [
        "#4E79A7", "#59A14F", "#E15759", "#B07AA1",
        "#F28E2B", "#76B7B2", "#EDC948", "#9C755F",
        "#FF9DA7", "#8CD17D", "#86BCB6", "#BAB0AC"
    ]

    return {
        n: palette[i % len(palette)]
        for i, n in enumerate(n_list)
    }

# =========================
# PLOTTING
# =========================

for category, k_data in results.items():

    ks = sorted(k_data.keys())

    n_types = sorted(
        set(
            n
            for k in ks
            for n in k_data[k].keys()
        ),
        key=lambda x: (
            x != "original",
            int(x.replace("u", "")) if x != "original" else -1
        )
    )

    color_map = generate_colors(n_types)

    x = np.arange(len(ks)) * 1.2

    width = 1.0 / len(n_types)

    fig, ax = plt.subplots(figsize=(18, 11))

    max_val = 0

    bar_containers = []

    for i, n in enumerate(n_types):

        vals = [
            k_data[k].get(n, (0, 0))[0]
            for k in ks
        ]

        bars = ax.bar(
            x + (i - len(n_types)/2) * width + width/2,
            vals,
            width=width,
            color=color_map[n],
            edgecolor="black",
            linewidth=0.6,
            label=n
        )

        bar_containers.append((bars, vals))

        max_val = max(
            max_val,
            max(vals)
        )

    # =========================
    # STYLE
    # =========================

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    ax.yaxis.grid(
        True,
        linestyle="--",
        linewidth=0.6,
        alpha=0.6
    )

    ax.set_title(
        f"NMF - {category}",
        fontsize=40,
        fontweight="bold",
        pad=20
    )

    ax.set_xlabel(
        "k (recommendation size)",
        fontsize=38,
        labelpad=14
    )

    ax.set_ylabel(
        "Avg count per user",
        fontsize=38,
        labelpad=14
    )

    ax.set_xticks(x)

    ax.set_xticklabels(
        ks,
        fontsize=34
    )

    ax.tick_params(
        axis='y',
        labelsize=34
    )

    ax.legend(
        fontsize=22,
        ncol=3
    )

    # =========================
    # BAR LABELS
    # =========================

    for bars, vals in bar_containers:

        for bar, val in zip(bars, vals):

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max_val * 0.015,
                f"{val:.2f}",
                ha="center",
                va="bottom",
                fontsize=22,
                rotation=90,
            )

    ax.set_ylim(
        top=max_val * 1.25
    )

    # =========================
    # SAVE FIGURE
    # =========================

    safe_name = (
        category.lower()
        .replace(' ', '_')
        .replace('&', 'and')
    )

    genre_folder = (
        OUT_BASE
        / f"nmf_{safe_name}_analysis"
    )

    genre_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    fig_path = (
        genre_folder
        / f"NMF - {category}.png"
    )

    plt.tight_layout()

    plt.savefig(
        fig_path,
        dpi=300
    )

    plt.close()

    # =========================
    # SAVE EXPLANATION TEXT
    # =========================

    explanation_path = (
        genre_folder
        / f"NMF - {category}.txt"
    )

    with open(explanation_path, "w") as f:

        f.write(
            f"NMF Analysis for Category: {category}\n\n"
        )

        f.write(
            "Metric: Average count of target category per user\n\n"
        )

        for k in ks:

            f.write(f"k = {k}\n")

            for n in n_types:

                val, users = k_data[k].get(
                    n,
                    (0, 0)
                )

                f.write(
                    f"  {n}: {val:.4f} (users={users})\n"
                )

            f.write("\n")

print("All NMF figures and explanation files saved.")

ORIGINAL_35recommendation.csv columns: ['user_id', 'book_id', 'est_score', 'rank', 'genre_g1', 'genre_g2', 'genres_all']
Known categories: {'Casual', 'Word', 'Video Players & Editors', 'Casino', 'Sports', 'Trivia', 'Adventure', 'Lifestyle', 'Business', 'Medical', 'Travel & Local', 'Board', 'Personalization', 'Strategy', 'Role Playing', 'Social', 'Auto & Vehicles', 'Entertainment', 'Weather', 'Food & Drink', 'Finance', 'House & Home', 'Dating', 'Arcade', 'Card', 'Tools', 'Comics', 'Music', 'Beauty', 'Communication', 'Libraries & Demo', 'News & Magazines', 'Maps & Navigation', 'Music & Audio', 'Parenting', 'Education', 'Events', 'Health & Fitness', 'Simulation', 'Photography', 'Shopping', 'Puzzle', 'Productivity', 'Art & Design', 'Educational', 'Action', 'Books & Reference', 'Racing'}
Failed reading f_Shopping_60u_pos5_neg1_allapps_25recommendation.csv: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.


### param

In [5]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Paths ──
BASE = "/home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result"
BI_DIR = os.path.join(BASE, "bi")
MU_DIR = os.path.join(BASE, "mio")  # Note: using mio for MU
BU_DIR = os.path.join(BASE, "bu")

# ── Root figures folder ──
DETAILS_FIG_DIR = os.path.join(BASE, "0519_NMF_details_figures")
os.makedirs(DETAILS_FIG_DIR, exist_ok=True)

# ── Patterns ──
BI_PATTERN = re.compile(r"^f_(.+?)_(\d+)u_pos5_neg1_allapps_bi_per_genre\.csv$")

# ── Step 1: Discover bi files ──
bi_files = {}
for fname in os.listdir(BI_DIR):
    m = BI_PATTERN.match(fname)
    if m:
        genre_key = m.group(1)
        n = int(m.group(2))
        bi_files[(genre_key, n)] = fname

print(f"Found {len(bi_files)} bi files")

# ── Step 2: Load and merge in memory only ──
merged_data = {}
bu_data = {}

for (genre_key, n), bi_fname in sorted(bi_files.items()):
    mu_fname = bi_fname.replace("_bi_", "_mu_")
    bu_fname = bi_fname.replace("_bi_", "_bu_").replace("_per_genre.csv", "_per_genre_biased.csv")

    bi_path = os.path.join(BI_DIR, bi_fname)
    mu_path = os.path.join(MU_DIR, mu_fname)
    bu_path = os.path.join(BU_DIR, bu_fname)

    if not (os.path.exists(mu_path) and os.path.exists(bu_path)):
        print(f"WARNING: Missing mu/bu for {bi_fname}, skipping.")
        continue

    df_bi = pd.read_csv(bi_path)
    df_mu = pd.read_csv(mu_path)
    df_bu = pd.read_csv(bu_path)

    for df_name, df in [("bi", df_bi), ("mu", df_mu), ("bu", df_bu)]:
        if "genre" in df.columns:
            df["genre"] = df["genre"].astype(str).str.strip()
            df = df[df["genre"].str.lower() != "unknown"].copy()

        if df_name == "bi":
            df_bi = df
        elif df_name == "mu":
            df_mu = df
        else:
            df_bu = df

    bu_data[(genre_key, n)] = df_bu.copy()

    if "user_type" in df_bu.columns:
        df_bu_merge = df_bu[df_bu["user_type"] == "non_biasing"].copy()
        if df_bu_merge.empty:
            df_bu_merge = df_bu.copy()
    else:
        df_bu_merge = df_bu.copy()

    merge_cols = ["genre"]
    if "is_target" in df_bi.columns:
        merge_cols.append("is_target")

    df = pd.merge(df_mu, df_bi, on=merge_cols, how="outer")
    df = pd.merge(df, df_bu_merge, on=merge_cols, how="outer")

    merged_data[(genre_key, n)] = df

print(f"Merged {len(merged_data)} configs")

# ── Step 3: Organize by genre ──
genre_groups = defaultdict(dict)
for (genre_key, n), df in merged_data.items():
    genre_groups[genre_key][n] = df

bu_groups = defaultdict(dict)
for (genre_key, n), df in bu_data.items():
    bu_groups[genre_key][n] = df

# ── Style ──
plt.style.use("seaborn-v0_8-pastel")

TARGET_COLOR = "#5B3A29"
NON_TARGET_LINEWIDTH = 1.5
TARGET_LINEWIDTH = 3.5

PASTEL_COLORS = [
    "#AEC6CF", "#FFD1DC", "#B5EAD7", "#FFDAC1", "#C3B1E1",
    "#F3E5AB", "#FFB7B2", "#B0E0E6", "#D5AAFF", "#FFFACD",
    "#E0BBE4", "#957DAD", "#D4E157", "#80CBC4", "#F48FB1",
]

TARGET_BIAS_COLOR = "#8B0000"
TARGET_NONBIAS_COLOR = "#5B3A29"
NONTARGET_BIAS_COLOR = "#1F77B4"
NONTARGET_NONBIAS_COLOR = "#7FB3D5"

# ── Helpers ──
def get_single_value(row, col):
    if row.empty or col not in row.columns:
        return None
    return row[col].values[0]


def get_genre_output_dir(genre_key):
    genre_dir = os.path.join(DETAILS_FIG_DIR, genre_key)
    os.makedirs(genre_dir, exist_ok=True)
    return genre_dir


def add_horizontal_legend(ax):
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=6,
        fontsize=10,
        frameon=True
    )


def plot_figure(genre_key, sorted_ns, n_dict, target_genre_name, metric_name, compute_value, ylabel, suffix):
    all_genres = set()
    for n in sorted_ns:
        if "genre" in n_dict[n].columns:
            all_genres.update(n_dict[n]["genre"].dropna().tolist())
    all_genres = sorted(all_genres)

    genre_series = {}
    for genre_name in all_genres:
        pts = []
        for n in sorted_ns:
            df = n_dict[n]
            row = df[df["genre"] == genre_name]
            if not row.empty:
                val = compute_value(row)
                if val is not None:
                    pts.append((n, val))
        genre_series[genre_name] = pts

    non_target = [g for g in all_genres if g != target_genre_name]

    color_map = {}
    for i, g in enumerate(non_target):
        color_map[g] = PASTEL_COLORS[i % len(PASTEL_COLORS)]
    color_map[target_genre_name] = TARGET_COLOR

    positions = list(range(len(sorted_ns)))
    n_to_pos = {n: p for n, p in zip(sorted_ns, positions)}

    fig, ax = plt.subplots(figsize=(14, 8))

    for g in non_target:
        pts = genre_series[g]
        if pts:
            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]
            ax.plot(xs, ys, marker="o", color=color_map[g],
                    linewidth=NON_TARGET_LINEWIDTH, markersize=9, alpha=0.8, label=g)

    pts = genre_series.get(target_genre_name, [])
    if pts:
        xs = [n_to_pos[x] for x, _ in pts]
        ys = [y for _, y in pts]
        ax.plot(xs, ys, marker="o", color=TARGET_COLOR,
                linewidth=TARGET_LINEWIDTH, markersize=7,
                zorder=10, label=f"{target_genre_name} (target)")

    pretty = genre_key.replace("_", " ")

    ax.set_title(f"NMF - {pretty} ({metric_name})", fontsize=26, fontweight="bold")
    ax.set_xlabel("Number of Fictitious Users (n)", fontsize=25)
    ax.set_ylabel(ylabel, fontsize=28, fontweight="bold")

    ax.set_xticks(positions)
    ax.set_xticklabels([str(n) for n in sorted_ns], rotation=45)

    add_horizontal_legend(ax)

    ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.12, 1, 1])

    genre_dir = get_genre_output_dir(genre_key)
    out = os.path.join(genre_dir, f"{suffix}.png")

    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out}")


def plot_bu_bias_comparison(genre_key, sorted_ns, bu_n_dict, ylabel, suffix):
    positions = list(range(len(sorted_ns)))
    n_to_pos = {n: p for n, p in zip(sorted_ns, positions)}

    target_bias_pts = []
    target_nonbias_pts = []
    non_target_bias_pts = []
    non_target_nonbias_pts = []

    for n in sorted_ns:
        df = bu_n_dict[n].copy()

        if "genre" in df.columns:
            df["genre"] = df["genre"].astype(str).str.strip()
            df = df[df["genre"].str.lower() != "unknown"]

        if "user_type" in df.columns:
            df["user_type"] = df["user_type"].astype(str).str.strip()

        if "is_target" in df.columns:
            df["is_target"] = df["is_target"].astype(str).str.strip()

        row = df[(df["is_target"] == "target") & (df["user_type"] == "biasing")]
        if not row.empty:
            target_bias_pts.append((n, row["avg_bu"].mean()))

        row = df[(df["is_target"] == "target") & (df["user_type"] == "non_biasing")]
        if not row.empty:
            target_nonbias_pts.append((n, row["avg_bu"].mean()))

        row = df[(df["is_target"] == "non_target") & (df["user_type"] == "biasing")]
        if not row.empty:
            non_target_bias_pts.append((n, row["avg_bu"].mean()))

        row = df[(df["is_target"] == "non_target") & (df["user_type"] == "non_biasing")]
        if not row.empty:
            non_target_nonbias_pts.append((n, row["avg_bu"].mean()))

    fig, ax = plt.subplots(figsize=(14, 8))

    series_info = [
        ("Target - biasing", target_bias_pts, TARGET_BIAS_COLOR, 3.5, "o"),
        ("Target - non_biasing", target_nonbias_pts, TARGET_NONBIAS_COLOR, 3.5, "s"),
        ("Non-target - biasing", non_target_bias_pts, NONTARGET_BIAS_COLOR, 2.5, "^"),
        ("Non-target - non_biasing", non_target_nonbias_pts, NONTARGET_NONBIAS_COLOR, 2.5, "D"),
    ]

    for label, pts, color, linewidth, marker in series_info:
        if pts:
            xs = [n_to_pos[x] for x, _ in pts]
            ys = [y for _, y in pts]
            ax.plot(xs, ys, marker=marker, color=color,
                    linewidth=linewidth, markersize=8, label=label)

    pretty = genre_key.replace("_", " ")

    ax.set_title(f"NMF - {pretty} (bᵤ biased vs unbiased)", fontsize=26, fontweight="bold")
    ax.set_xlabel("Number of Fictitious Users (n)", fontsize=25)
    ax.set_ylabel(ylabel, fontsize=28, fontweight="bold")

    ax.set_xticks(positions)
    ax.set_xticklabels([str(n) for n in sorted_ns], rotation=45)

    add_horizontal_legend(ax)

    ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0.12, 1, 1])

    genre_dir = get_genre_output_dir(genre_key)
    out = os.path.join(genre_dir, f"{suffix}.png")

    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out}")


# ── Step 4: Generate ALL figures ──
for genre_key, n_dict in sorted(genre_groups.items()):
    sorted_ns = sorted(n_dict.keys())
    if not sorted_ns:
        continue

    sample_df = n_dict[sorted_ns[0]]
    if "is_target" not in sample_df.columns:
        continue

    target_rows = sample_df[sample_df["is_target"] == "target"]
    if target_rows.empty:
        continue

    target_genre_name = target_rows["genre"].iloc[0]

    plot_figure(genre_key, sorted_ns, n_dict, target_genre_name,
                "μ + bᵢ",
                lambda r: (get_single_value(r, "global_mean_mu") + get_single_value(r, "avg_bi")
                           if get_single_value(r, "global_mean_mu") is not None and get_single_value(r, "avg_bi") is not None else None),
                r"$\mu + b_i$", "mu_plus_bi")

    plot_figure(genre_key, sorted_ns, n_dict, target_genre_name,
                "bᵢ",
                lambda r: get_single_value(r, "avg_bi"),
                r"$b_i$", "bi_only")

    plot_figure(genre_key, sorted_ns, n_dict, target_genre_name,
                "bᵤ (non-biasing users)",
                lambda r: get_single_value(r, "avg_bu"),
                r"$b_u$", "bu_only_non_biasing")

    plot_figure(genre_key, sorted_ns, n_dict, target_genre_name,
                "μ",
                lambda r: get_single_value(r, "global_mean_mu"),
                r"$\mu$", "mu_only")

    if genre_key in bu_groups:
        plot_bu_bias_comparison(
            genre_key,
            sorted_ns,
            bu_groups[genre_key],
            r"$b_u$",
            "bu_biased_vs_unbiased_target_vs_nontarget"
        )

print("Done! Figures saved per genre folder.")

Found 8 bi files
Merged 8 configs
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Events/mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Events/bi_only.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Events/bu_only_non_biasing.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Events/mu_only.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Events/bu_biased_vs_unbiased_target_vs_nontarget.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Shopping/mu_plus_bi.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current/0519/0519_NMF/result/0519_NMF_details_figures/Shopping/bi_only.png
Saved: /home/moshtasa/Research/phd-recsys/Mobile/current

In [3]:
# ── SANITY CHECKS ──

print("\n========== SANITY CHECK ==========")
print(f"BASE exists: {os.path.exists(BASE)}")
print(f"BI_DIR exists: {os.path.exists(BI_DIR)}")
print(f"MU_DIR exists: {os.path.exists(MU_DIR)}")
print(f"BU_DIR exists: {os.path.exists(BU_DIR)}")

print("\n========== BI DIRECTORY SAMPLE FILES ==========")
if os.path.exists(BI_DIR):
    bi_dir_files = sorted(os.listdir(BI_DIR))
    print(f"Total files in BI_DIR: {len(bi_dir_files)}")

    for f in bi_dir_files[:10]:
        print(f)

print("\n========== TEST REGEX MATCHING ==========")

if os.path.exists(BI_DIR):
    for f in bi_dir_files[:10]:
        m = BI_PATTERN.match(f)

        print(f"\nFILE: {f}")
        print(f"MATCH: {m is not None}")

        if m:
            print(f"genre_key = {m.group(1)}")
            print(f"n = {m.group(2)}")

print("\n========== PATTERN ==========")
print(BI_PATTERN.pattern)

print("\n========== CHECK ONE CSV HEAD ==========")

if os.path.exists(BI_DIR):
    matched_files = []

    for f in bi_dir_files:
        if BI_PATTERN.match(f):
            matched_files.append(f)

    print(f"Matched files count = {len(matched_files)}")

    if matched_files:
        sample_file = matched_files[0]

        sample_path = os.path.join(BI_DIR, sample_file)

        print(f"\nReading: {sample_file}")

        df_test = pd.read_csv(sample_path)

        print("\nHEAD:")
        print(df_test.head())

        print("\nCOLUMNS:")
        print(df_test.columns.tolist())

        print("\nDTYPES:")
        print(df_test.dtypes)

        print("\nUNIQUE is_target:")
        if "is_target" in df_test.columns:
            print(df_test["is_target"].unique())

        print("\nUNIQUE genre sample:")
        if "genre" in df_test.columns:
            print(df_test["genre"].dropna().unique()[:10])

print("\n========== STYLE CHECK ==========")
print("Current matplotlib style:")
print(plt.rcParams["axes.facecolor"])

print("\n========== END SANITY CHECK ==========")


========== SANITY CHECK ==========
BASE exists: True
BI_DIR exists: True
MU_DIR exists: True
BU_DIR exists: True

========== BI DIRECTORY SAMPLE FILES ==========
Total files in BI_DIR: 9
ORIGINAL_bi_per_genre.csv
f_Events_30u_pos5_neg1_allapps_bi_per_genre.csv
f_Events_60u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_2000u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_300u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_30u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_500u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_60u_pos5_neg1_allapps_bi_per_genre.csv
f_Shopping_7000u_pos5_neg1_allapps_bi_per_genre.csv

========== TEST REGEX MATCHING ==========

FILE: ORIGINAL_bi_per_genre.csv
MATCH: False

FILE: f_Events_30u_pos5_neg1_allapps_bi_per_genre.csv
MATCH: False

FILE: f_Events_60u_pos5_neg1_allapps_bi_per_genre.csv
MATCH: False

FILE: f_Shopping_2000u_pos5_neg1_allapps_bi_per_genre.csv
MATCH: False

FILE: f_Shopping_300u_pos5_neg1_allapps_bi_per_genre.csv
MATCH: False

FILE: f_Shoppin